In [1]:
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

Device: cuda


In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import sys
sys.path.append('C:/Users/rishe/Dissertation')
import pandas as pd
from utils.data_utils.data_helper_utils import temporal_split
from utils.data_utils.dataset_files.dataset_with_meta_features import RainfallWindowDataset
from utils.loss import SeasonalFocalMSELoss
from models.lstm_with_seasonal_criterion import RainfallLSTM, train_model, evaluate_on_test, collect_test_predictions
print('Imports OK')

Imports OK


In [4]:
# Load data and split
DATA_PATH = 'C:/Users/rishe/Dissertation/data/processed_rain.parquet'
df = pd.read_parquet(DATA_PATH)
df_train, df_val, df_test = temporal_split(df)
print('Splits:', len(df_train), len(df_val), len(df_test))

Splits: 3319026 711259 711380


In [5]:
# Build datasets
L = 30  # window length
H = 1   # horizon
train_ds = RainfallWindowDataset(df_train, window_length=L, horizon=H, min_days_per_station=L+H)
val_ds = RainfallWindowDataset(df_val, window_length=L, horizon=H, min_days_per_station=L+H)
test_ds = RainfallWindowDataset(df_test, window_length=L, horizon=H, min_days_per_station=L+H)
print('Dataset sizes ->', len(train_ds), len(val_ds), len(test_ds))

Dataset sizes -> 3310236 702469 702590


In [6]:
# Create model and seasonal loss
HIDDEN = 128
model = RainfallLSTM(input_dim=train_ds.X.shape[-1], hidden_dim=HIDDEN).to(device)
criterion = SeasonalFocalMSELoss(gamma=2.0, monsoon_weight=3.0, non_monsoon_weight=1.0, reduction='mean')
print('Model and criterion ready')

Model and criterion ready


In [7]:
# Quick training run (small epochs)
trained = train_model(
    train_ds=train_ds,
    val_ds=val_ds,
    model=model,
    device=device,
    batch_size=64,
    epochs=10,
    lr=1e-3,
    criterion=criterion,
    save_dir='C:/Users/rishe/Dissertation/experiments/saved_models/exp_5_seasonal',
    log_dir='C:/Users/rishe/Dissertation/experiments/logs/exp_5_seasonal',
    experiment_name='exp_5_seasonal'
)
print('Training finished')

2026-02-08 23:02:25 | INFO | Starting training
2026-02-08 23:02:25 | INFO | Device: cuda


Epochs:   0%|          | 0/10 [00:00<?, ?it/s]

2026-02-08 23:04:53 | INFO | Epoch 001 | Train Loss: nan | Val Loss: nan
2026-02-08 23:04:53 | INFO | Checkpoint saved: C:/Users/rishe/Dissertation/experiments/saved_models/exp_5_seasonal\epoch_1.pt
2026-02-08 23:07:34 | INFO | Epoch 002 | Train Loss: nan | Val Loss: nan
2026-02-08 23:07:34 | INFO | Checkpoint saved: C:/Users/rishe/Dissertation/experiments/saved_models/exp_5_seasonal\epoch_2.pt


KeyboardInterrupt: 

In [ ]:
# Evaluate on test set
metrics = evaluate_on_test(test_ds, trained, device)
print('Test metrics:', metrics)
df_eval = collect_test_predictions(test_ds, trained, device)
df_eval.head()

In [ ]:
# Imports needed for evaluation and plotting
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from utils.metric_utils.metrics import rmse, mae, bias, nrmse, seasonal_subset, stationwise_metrics

print('Evaluation imports OK')

In [ ]:
# Compute basic error and masks for analysis
df_eval['error'] = df_eval['y'] - df_eval['yhat']
rainy_mask = df_eval['y'] > 0
non_rainy_mask = df_eval['y'] == 0

monsoon_months = [6, 7, 8, 9]
non_monsoon_months = [1, 2, 3, 4, 5, 10, 11, 12]
monsoon_mask = df_eval['month'].isin(monsoon_months)
non_monsoon_mask = df_eval['month'].isin(non_monsoon_months)

print('Error and masks computed')

In [ ]:
# Overall evaluation metrics
overall_metrics = {
    'RMSE': rmse(df_eval.y.values, df_eval.yhat.values),
    'MAE': mae(df_eval.y.values, df_eval.yhat.values),
    'Bias': bias(df_eval.y.values, df_eval.yhat.values),
    'NRMSE': nrmse(df_eval.y.values, df_eval.yhat.values),
}
print('
Overall Evaluation Metrics (Seasonal Focal Loss):')
print(pd.DataFrame(overall_metrics, index=['Seasonal Focal Loss Model']).round(4))

In [ ]:
# Seasonal evaluation
df_monsoon = seasonal_subset(df_eval, monsoon_months)
df_non_monsoon = seasonal_subset(df_eval, non_monsoon_months)

seasonal_metrics = pd.DataFrame({
    'Monsoon': {
        'RMSE': rmse(df_monsoon.y, df_monsoon.yhat),
        'MAE': mae(df_monsoon.y, df_monsoon.yhat),
        'Bias': bias(df_monsoon.y, df_monsoon.yhat),
    },
    'Non-Monsoon': {
        'RMSE': rmse(df_non_monsoon.y, df_non_monsoon.yhat),
        'MAE': mae(df_non_monsoon.y, df_non_monsoon.yhat),
        'Bias': bias(df_non_monsoon.y, df_non_monsoon.yhat),
    }
}).T

print('
Seasonal Performance:')
print(seasonal_metrics.round(4))

In [ ]:
# Per-station metrics
df_station_metrics = stationwise_metrics(df_eval)

print('
Station-wise metrics summary:')
print(df_station_metrics.describe().round(4))

In [ ]:
# Month-wise RMSE
month_rmse = (
    df_eval
    .groupby('month')
    .apply(lambda g: rmse(g.y.values, g.yhat.values))
    .sort_index()
)

plt.figure(figsize=(7, 4))
plt.plot(month_rmse.index, month_rmse.values, marker='o', linewidth=2)
plt.xticks(range(1, 13))
plt.xlabel('Month')
plt.ylabel('RMSE (mm)')
plt.title('Month-wise RMSE (Seasonal Focal Loss Model)')
plt.grid(True, which='both', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

In [ ]:
# Error distribution statistics and plot
print('Error Statistics:')
print((df_eval['error']).describe(percentiles=[0.1, .2, .25, .3, .4, .5, .6, .7, .75, .8, .9, .95]).round(4))

plt.figure(figsize=(6, 4))
sns.histplot(df_eval['error'], bins=50, kde=True)
plt.xlabel('Prediction Error (ŷ − y)')
plt.title('Error Distribution (Seasonal Focal Loss Model)')
plt.tight_layout()
plt.show()

In [ ]:
# Rainy vs Non-Rainy error distributions
plt.figure(figsize=(7, 4))
sns.histplot(df_eval.loc[rainy_mask, 'error'], bins=50, kde=True, color='tab:blue', label='Rainy days (y > 0)', stat='density')
sns.histplot(df_eval.loc[non_rainy_mask, 'error'], bins=50, kde=True, color='tab:orange', label='Non-rainy days (y = 0)', stat='density')
plt.axvline(0, color='black', linestyle='--', linewidth=1)
plt.xlabel('Prediction Error (ŷ − y)')
plt.title('Error Distribution: Rainy vs Non-Rainy Days')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Monsoon vs Non-Monsoon error distributions
plt.figure(figsize=(7, 4))
sns.histplot(df_eval.loc[monsoon_mask, 'error'], bins=50, kde=True, color='tab:green', label='Monsoon', stat='density')
sns.histplot(df_eval.loc[non_monsoon_mask, 'error'], bins=50, kde=True, color='tab:red', label='Non-Monsoon', stat='density')
plt.axvline(0, color='black', linestyle='--', linewidth=1)
plt.xlabel('Prediction Error (ŷ − y)')
plt.title('Error Distribution: Monsoon vs Non-Monsoon')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Conditional RMSE by rainfall regimes
def conditional_rmse(mask, name):
    e = df_eval.loc[mask]
    return {
        'Regime': name,
        'RMSE': rmse(e.y, e.yhat),
        'MAE': mae(e.y, e.yhat),
        'Bias': bias(e.y, e.yhat),
        'Count': len(e)
    }

conditional_metrics = pd.DataFrame([
    conditional_rmse(df_eval.y == 0, 'Non-rainy'),
    conditional_rmse((df_eval.y > 0) & (df_eval.y <= 10), 'Light rain'),
    conditional_rmse((df_eval.y > 10) & (df_eval.y <= 50), 'Moderate rain'),
    conditional_rmse(df_eval.y > 50, 'Heavy rain'),
] )

print('
Conditional RMSE by Rainfall Regime:')
print(conditional_metrics.round(4))

In [ ]:
# Baseline comparisons
yhat_zero = np.zeros_like(df_eval.y)
yhat_mean = np.full_like(df_eval.y, df_eval.y.mean())

baseline_table = pd.DataFrame({
    'Seasonal Focal Loss Model': {
        'RMSE': rmse(df_eval.y, df_eval.yhat),
        'MAE': mae(df_eval.y, df_eval.yhat),
    },
    'Zero Predictor': {
        'RMSE': rmse(df_eval.y, yhat_zero),
        'MAE': mae(df_eval.y, yhat_zero),
    },
    'Mean Predictor': {
        'RMSE': rmse(df_eval.y, yhat_mean),
        'MAE': mae(df_eval.y, yhat_mean),
    },
}).T

print('
Baseline Comparison:')
print(baseline_table.round(4))

In [ ]:
# Scatter plot: True vs Predicted
plt.figure(figsize=(7, 5))
plt.scatter(df_eval['y'], df_eval['yhat'], alpha=0.3, s=10)
max_val = max(df_eval['y'].max(), df_eval['yhat'].max())
plt.plot([0, max_val], [0, max_val], 'r--', linewidth=2, label='Perfect prediction')
plt.xlabel('True Rainfall (mm)')
plt.ylabel('Predicted Rainfall (mm)')
plt.title('True vs Predicted Rainfall (Seasonal Focal Loss Model)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Summary
summary_data = [
    {'Metric': 'RMSE', 'Value': overall_metrics['RMSE']},
    {'Metric': 'MAE', 'Value': overall_metrics['MAE']},
    {'Metric': 'Bias', 'Value': overall_metrics['Bias']},
    {'Metric': 'NRMSE', 'Value': overall_metrics['NRMSE']},
]

print('
' + '='*60)
print('EXPERIMENT SUMMARY: Seasonal Focal MSE Loss')
print('='*60)
print(f'Window Length: {L} days')
print(f'Horizon: {H} day')
print(f'Batch Size: 64')
print(f'Epochs: 5')
print(f'Learning Rate: {1e-3}')
print(f'Hidden Dimension: {HIDDEN}')
print('
Results:')
print(pd.DataFrame(summary_data).to_string(index=False))
print('='*60)